# 06 — RecBole models (LightGCN / GRU4Rec / SASRec)

Requires `artifacts/sample.parquet` and a GPU. Install with:
```
pip install -e ".[recbole]"
```
plus a CUDA-enabled PyTorch build.

Trains each model on our leave-last-N split and scores via the shared `evaluate_predictions`.

**Note: RecBole's API is version-sensitive — validate these calls against the installed RecBole when you run it.**

In [ ]:
!pip install -q recbole

In [ ]:
import os, pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.schema import USER, BOOK
from book_recsys.eval.harness import build_relevance, evaluate_predictions
from book_recsys.recbole_adapter.atomic import write_inter_file
from book_recsys.recbole_adapter.export import recbole_predictions

In [ ]:
sample = pd.read_parquet("artifacts/sample.parquet")
train, holdout = leave_last_n_out(sample, n=1)
relevance = build_relevance(holdout)

DATASET = "goodreads"
ds_dir = f"recbole_data/{DATASET}"
os.makedirs(ds_dir, exist_ok=True)
write_inter_file(train, f"{ds_dir}/{DATASET}.inter")

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer
from recbole.utils.case_study import full_sort_topk

def run_model(model_name, extra=None):
    cfg = {
        "data_path": "recbole_data", "dataset": DATASET,
        "USER_ID_FIELD": "user_id", "ITEM_ID_FIELD": "item_id",
        "RATING_FIELD": "rating", "TIME_FIELD": "timestamp",
        "load_col": {"inter": ["user_id", "item_id", "rating", "timestamp"]},
        "epochs": 20, "topk": 10,
        "eval_args": {"split": {"RS": [0.9, 0.0, 0.1]}, "order": "TO"},
        "metrics": ["NDCG", "Recall", "MRR"], "valid_metric": "NDCG@10",
    }
    cfg.update(extra or {})
    config = Config(model=model_name, dataset=DATASET, config_dict=cfg)
    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
    trainer.fit(train_data, valid_data)

    internal_users = list(range(1, dataset.user_num))  # skip [PAD]=0
    _, topk_iid = full_sort_topk(internal_users, model, test_data, k=10,
                                 device=config["device"])
    uid2token = {i: dataset.id2token(dataset.uid_field, i) for i in internal_users}
    iid2token = {int(i): dataset.id2token(dataset.iid_field, int(i))
                 for i in topk_iid.reshape(-1).unique().tolist()}
    preds = recbole_predictions(internal_users, topk_iid.tolist(), uid2token, iid2token)
    return evaluate_predictions(preds, relevance, k=10)

results = {}
results["LightGCN"] = run_model("LightGCN")
results["GRU4Rec"] = run_model("GRU4Rec")
results["SASRec"] = run_model("SASRec")
pd.DataFrame(results).T

## Results

Results from the cell above feed the model report (later plan step).

**Caveat:** sequential models (GRU4Rec / SASRec) order interactions by timestamp; RecBole handles this via `eval_args.order="TO"`. Validate that RecBole's internal split aligns with our holdout when you run it.